In [2]:
import chromadb, pandas as pd
from chromadb.utils import embedding_functions

In [3]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="paraphrase-multilingual-MiniLM-L12-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16350.30it/s]


In [4]:
cliente = chromadb.PersistentClient(path="./bdv")

In [5]:
col_productos = cliente.get_or_create_collection(
    name = "productos",
    embedding_function = ef,
    metadata = {"hnsw:space": "cosine"}
)

In [7]:
df_productos = pd.read_csv("data/productos.csv")

In [12]:
df_productos.describe()

,id,precio,stock,calificacion_promedio,total_resenas
count,90.000000,90.000000,90.000000,90.000000,90.000000
mean,45.500000,2326.444444,23.577778,3.535556,3.533333
std,26.124701,2464.095231,14.886068,0.701253,0.863752
min,1.000000,59.000000,3.000000,1.800000,1.000000
25%,23.250000,411.500000,12.000000,3.000000,3.000000
50%,45.500000,1649.000000,20.000000,3.450000,4.000000
75%,67.750000,3299.000000,30.000000,4.000000,4.000000
max,90.000000,12999.000000,80.000000,5.000000,5.000000


In [14]:
col_productos.add(
    ids = df_productos["id"].astype(str).tolist(),
    documents = (df_productos["nombre"] + ". " + df_productos["descripcion"]).tolist(),
    metadatas = df_productos[["categoria", "precio"]].to_dict("records")
)

In [16]:
res = col_productos.query(query_texts=["celular con buena cámara"], n_results = 3)
res

{'ids': [['3', '7', '4']],
 'embeddings': None,
 'documents': [['Samsung Galaxy S24 Ultra 512GB. Smartphone con S Pen integrado, camara cuadruple de 200MP con zoom optico de 5x y pantalla Dynamic AMOLED 2X de 6.8 pulgadas a 120Hz. Incorpora marco de titanio, funciones de inteligencia artificial Galaxy AI y resistencia IP68.',
   'Google Pixel 8 128GB. Smartphone con chip Google Tensor G3 y camara computacional avanzada con Magic Editor para retocar fotos con inteligencia artificial. Ofrece pantalla Actua de 6.2 pulgadas a 120Hz y hasta 7 anos de actualizaciones de Android garantizadas.',
   'Samsung Galaxy A55 128GB. Smartphone de gama media con pantalla Super AMOLED de 6.6 pulgadas a 120Hz y camara principal de 50MP con estabilizacion optica. Cuenta con procesador Exynos 1480, certificacion IP67 y bateria de 5000mAh con carga rapida de 25W.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'categoria': 'Smartphones', 'precio': 529